# Part C: Sentiment Analysis

Project directions covered:
1. Compare the performance of different Transformer architectures
2. Deal with small datasets — compare multiple training sizes

**Models:** DistilBERT, BERT-base, RoBERTa  
**Dataset:** IMDB (Hugging Face)  
**Key principle:** Model selection uses *validation* F1 only — test set is touched once at the end.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers datasets accelerate pandas scikit-learn matplotlib seaborn

In [ ]:
# Cell 2 — Imports
import os
import json
import random
import shutil
from dataclasses import asdict, dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [ ]:
# Cell 3 — Config and reproducibility
SEED = 42
MAX_LEN = 256
BATCH_SIZE = 8
EPOCHS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# Small-dataset experiment: compare 3 training sizes
TRAIN_SIZES = [500, 1000, 2000]
VAL_SIZE = 1000
TEST_SIZE = 2000

# Three Transformer architectures to compare
MODEL_NAMES = [
    "distilbert-base-uncased",   # lightweight ~66M params
    "bert-base-uncased",          # classic BERT ~110M params
    "roberta-base",               # improved pretraining ~125M params
]

LABEL_NAMES = ["negative", "positive"]
OUTPUT_DIR = "part_c_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Fix all random seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
# Cell 4 — Load IMDB dataset and create splits
print("Loading IMDB dataset...")
raw_data = load_dataset("imdb")

# Use official train split for training only (no data leakage)
full_train = raw_data["train"].shuffle(seed=SEED)
full_test  = raw_data["test"].shuffle(seed=SEED)

# Val and test come from the held-out test split
val_data  = full_test.select(range(VAL_SIZE))
test_data = full_test.select(range(VAL_SIZE, VAL_SIZE + TEST_SIZE))

print(f"Validation size : {len(val_data):,}")
print(f"Test size       : {len(test_data):,}")

def get_train_subset(train_size: int):
    """Return the first train_size examples from the shuffled training set."""
    return full_train.select(range(train_size))

In [ ]:
# Cell 5 — Tokenisation helpers and metric function
def tokenize_dataset(dataset, tokenizer):
    """Tokenise a HuggingFace dataset; returns PyTorch-format tensors."""
    def tokenize(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
        )
    tokenized = dataset.map(tokenize, batched=True)
    tokenized = tokenized.remove_columns(["text"])
    tokenized = tokenized.rename_column("label", "labels")
    tokenized.set_format("torch")
    return tokenized


def compute_metrics(eval_pred):
    """Accuracy + binary F1 used by the HuggingFace Trainer during evaluation."""
    labels = eval_pred.label_ids
    preds  = np.argmax(eval_pred.predictions, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds),
    }


def build_training_args(output_dir: str):
    """Build TrainingArguments; handles eval_strategy rename across HF versions."""
    common = dict(
        output_dir                  = output_dir,
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "f1",
        greater_is_better           = True,
        learning_rate               = LEARNING_RATE,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE,
        num_train_epochs            = EPOCHS,
        weight_decay                = WEIGHT_DECAY,
        logging_steps               = 50,
        report_to                   = [],
        fp16                        = torch.cuda.is_available(),
        seed                        = SEED,
    )
    try:
        return TrainingArguments(eval_strategy="epoch", **common)
    except TypeError:
        return TrainingArguments(evaluation_strategy="epoch", **common)


print("Helpers defined.")

In [ ]:
# Cell 6 — RunResult dataclass
@dataclass
class RunResult:
    """Stores validation metrics for one (model, train_size) experiment."""
    model_name:   str
    train_size:   int
    val_accuracy: float
    val_f1:       float
    output_dir:   str

print("RunResult dataclass defined.")

In [ ]:
# Cell 7 — Single experiment: train one model at one training size
def run_experiment(model_name: str, train_size: int) -> RunResult:
    """Fine-tune model_name on train_size examples; return validation metrics."""
    print("=" * 80)
    print(f"Training  model={model_name}  train_size={train_size}")

    run_dir = os.path.join(
        OUTPUT_DIR, f"{model_name.replace('/', '_')}_train{train_size}"
    )

    tokenizer       = AutoTokenizer.from_pretrained(model_name)
    train_subset    = get_train_subset(train_size)
    tokenized_train = tokenize_dataset(train_subset, tokenizer)
    tokenized_val   = tokenize_dataset(val_data,     tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    trainer = Trainer(
        model           = model,
        args            = build_training_args(run_dir),
        train_dataset   = tokenized_train,
        eval_dataset    = tokenized_val,
        compute_metrics = compute_metrics,
    )
    trainer.train()

    # Use predict() to avoid the Trainer.evaluate() notebook-callback bug
    val_out   = trainer.predict(tokenized_val)
    val_preds = np.argmax(val_out.predictions, axis=1)
    val_true  = val_out.label_ids

    val_acc = accuracy_score(val_true, val_preds)
    val_f1  = f1_score(val_true, val_preds)

    print(f"Validation Accuracy: {val_acc:.4f}  |  Validation F1: {val_f1:.4f}")

    return RunResult(
        model_name   = model_name,
        train_size   = train_size,
        val_accuracy = val_acc,
        val_f1       = val_f1,
        output_dir   = run_dir,
    )

print("run_experiment() defined.")

In [ ]:
# Cell 8 — Run all experiments (3 models x 3 train sizes = 9 runs)
# Warning: long-running cell — approx 20-40 min on Colab T4 GPU
results: List[RunResult] = []

for model_name in MODEL_NAMES:
    for train_size in TRAIN_SIZES:
        result = run_experiment(model_name, train_size)
        results.append(result)

# Build summary sorted by validation F1 (NOT test — avoids data leakage)
summary_df = pd.DataFrame([asdict(r) for r in results])
summary_df = summary_df.sort_values(
    by=["val_f1", "val_accuracy"], ascending=False
).reset_index(drop=True)

summary_df.to_csv(os.path.join(OUTPUT_DIR, "summary.csv"), index=False)
print("\nValidation summary (sorted by val F1):")
summary_df

In [ ]:
# Cell 9 — Identify best configuration using validation F1 only
best_row        = summary_df.iloc[0]
best_model_name = best_row["model_name"]
best_train_size = int(best_row["train_size"])
best_run_dir    = best_row["output_dir"]

print("Best configuration selected by VALIDATION F1 — test set not yet touched")
print(best_row.to_string())

In [ ]:
# Cell 10 — Retrain best configuration, then evaluate on test set (once only)
print("\nRetraining best configuration before final test evaluation...")

best_tokenizer       = AutoTokenizer.from_pretrained(best_model_name)
tokenized_best_train = tokenize_dataset(get_train_subset(best_train_size), best_tokenizer)
tokenized_best_val   = tokenize_dataset(val_data,  best_tokenizer)
tokenized_test       = tokenize_dataset(test_data, best_tokenizer)

best_model = AutoModelForSequenceClassification.from_pretrained(best_model_name, num_labels=2)

best_trainer = Trainer(
    model           = best_model,
    args            = build_training_args(os.path.join(OUTPUT_DIR, "best_reloaded_run")),
    train_dataset   = tokenized_best_train,
    eval_dataset    = tokenized_best_val,
    compute_metrics = compute_metrics,
)
best_trainer.train()

# Final test evaluation — only happens once
test_out      = best_trainer.predict(tokenized_test)
y_pred        = np.argmax(test_out.predictions, axis=1)
y_true        = test_out.label_ids
test_accuracy = accuracy_score(y_true, y_pred)
test_f1       = f1_score(y_true, y_pred)
cm            = confusion_matrix(y_true, y_pred)
report        = classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=4)

print(f"\nFinal Test Accuracy : {test_accuracy:.4f}")
print(f"Final Test F1       : {test_f1:.4f}")
print("\nClassification Report:")
print(report)

In [ ]:
# Cell 11 — Save all metrics to disk
final_metrics = {
    "best_model_name": best_model_name,
    "best_train_size": best_train_size,
    "test_accuracy":   float(test_accuracy),
    "test_f1":         float(test_f1),
}

with open(os.path.join(OUTPUT_DIR, "final_test_metrics.json"), "w") as f:
    json.dump(final_metrics, f, indent=2)

with open(os.path.join(OUTPUT_DIR, "classification_report.txt"), "w") as f:
    f.write(report)

print("Saved: final_test_metrics.json")
print("Saved: classification_report.txt")

In [ ]:
# Cell 12 — Plot: Validation F1 vs training size for each model
plt.figure(figsize=(8, 5))
for model_name in MODEL_NAMES:
    subset = summary_df[summary_df["model_name"] == model_name].sort_values("train_size")
    plt.plot(subset["train_size"], subset["val_f1"], marker="o", label=model_name)

plt.title("Validation F1 vs Training Size")
plt.xlabel("Training Size")
plt.ylabel("Validation F1")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "val_f1_vs_train_size.png"), dpi=200)
plt.show()
print("Saved: val_f1_vs_train_size.png")

In [ ]:
# Cell 13 — Plot: Validation accuracy vs training size for each model
plt.figure(figsize=(8, 5))
for model_name in MODEL_NAMES:
    subset = summary_df[summary_df["model_name"] == model_name].sort_values("train_size")
    plt.plot(subset["train_size"], subset["val_accuracy"], marker="o", label=model_name)

plt.title("Validation Accuracy vs Training Size")
plt.xlabel("Training Size")
plt.ylabel("Validation Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "val_accuracy_vs_train_size.png"), dpi=200)
plt.show()
print("Saved: val_accuracy_vs_train_size.png")

In [ ]:
# Cell 14 — Plot: Final test confusion matrix
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
plt.title(f"Test Confusion Matrix\n({best_model_name}, train_size={best_train_size})")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "test_confusion_matrix.png"), dpi=200)
plt.show()
print("Saved: test_confusion_matrix.png")

In [ ]:
# Cell 15 — Inference demo using the best model
def predict_sentiment(text: str) -> Tuple[str, float]:
    """Run inference with the best fine-tuned model. Returns (label, confidence)."""
    best_trainer.model.eval()
    inputs = best_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LEN,
    )
    inputs = {k: v.to(best_trainer.model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs  = best_trainer.model(**inputs)
        probs    = torch.softmax(outputs.logits, dim=1)[0]
        pred_idx = int(torch.argmax(probs).item())

    return LABEL_NAMES[pred_idx], float(probs[pred_idx].item())


examples = [
    "The movie was unexpectedly brilliant and emotionally satisfying.",
    "This film was painfully dull and a complete waste of time.",
    "An absolute masterpiece — one of the best films I have ever seen.",
    "I walked out halfway through. Terrible pacing and wooden dialogue.",
]

print(f"Inference demo — best model: {best_model_name}\n")
for text in examples:
    label, confidence = predict_sentiment(text)
    print(f"[{label.upper():8s} {confidence:.2%}]  {text}")

print("\nAll outputs saved in:", OUTPUT_DIR)